# HRRR CAPE + 0–3 km SRH over Chicagoland — Argonne point analysis

Severe-weather environment from the **12Z HRRR** run of 2026-07-19, focused on the
Chicagoland area with Argonne National Laboratory as the point of interest.

**Products**
1. Map of surface **CAPE** (fill) with **0–3 km storm-relative helicity** (SRH) contours over IA/WI/IL/IN/MI, at the hour of maximum CAPE at Argonne.
2. Time series of CAPE & SRH at the Argonne gridcell (F00–F48).
3. **Skew-T / log-p** sounding at Argonne for the max-CAPE hour.
4. Time series of **composite reflectivity** at Argonne.
5. An **animation** stepping the composite through all forecast hours.

Data are pulled with [Herbie](https://herbie.readthedocs.io) (byte-range subsets from the
NOAA HRRR AWS archive). Colormaps are from [cmweather](https://github.com/openradar/cmweather).

*Co-authored by Scott Collis and Claude.*

In [ ]:
import os, numpy as np, pandas as pd, warnings
import matplotlib.pyplot as plt, matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy.ndimage import gaussian_filter
import cartopy, cartopy.crs as ccrs, cartopy.feature as cfeature
from herbie import Herbie
from metpy.plots import SkewT, USCOUNTIES
from metpy.units import units
import metpy.calc as mpcalc
import cmweather  # registers Carbone/Homeyer/... colormaps
warnings.filterwarnings("ignore")

# Keep Herbie + cartopy caches inside the working dir (sandbox-friendly).
SAVE = os.path.join(os.getcwd(), "herbie_data"); os.makedirs(SAVE, exist_ok=True)
_cdir = os.path.join(os.getcwd(), "cartopy_data"); os.makedirs(_cdir, exist_ok=True)
cartopy.config["data_dir"] = _cdir; cartopy.config["pre_existing_data_dir"] = _cdir
os.environ["CARTOPY_DATA_DIR"] = _cdir

## Configuration

In [ ]:
RUN_INIT   = pd.Timestamp("2026-07-19 12:00")   # HRRR cycle (UTC)
FHOURS     = list(range(0, 49))                  # F00–F48
ARGONNE_LAT, ARGONNE_LON = 41.7183, -87.9786     # Argonne National Laboratory
CDT_OFFSET = pd.Timedelta(hours=5)               # UTC -> CDT (July = CDT)

# Map domain: IA/WI  through  IL/IN/MI  (E–W wide Chicagoland view)
DOMAIN = dict(lon_min=-91.8, lon_max=-84.2, lat_min=40.2, lat_max=43.4)

# Focal colors + colormap
CMAP_CAPE = "HomeyerRainbow"
C_CAPE, C_SRH, C_REFC = "#c0392b", "#2471a3", "#117a3d"

## Helper: open a single HRRR field as a 2-D array

In [ ]:
_cache = {}
def open_field(fxx, search, var=None, product="sfc"):
    '''Open one HRRR field for forecast hour `fxx`. Returns an xarray Dataset.'''
    key = (fxx, search, product)
    if key not in _cache:
        H = Herbie(RUN_INIT, model="hrrr", product=product, fxx=fxx, save_dir=SAVE, verbose=False)
        _cache[key] = H.xarray(search, remove_grib=False)
    return _cache[key]

## Locate the Argonne gridcell

In [ ]:
d0 = open_field(0, ":CAPE:surface:")
lat = d0.latitude.values; lon = np.where(d0.longitude.values > 180, d0.longitude.values - 360, d0.longitude.values)
dist = (lat - ARGONNE_LAT)**2 + (lon - ARGONNE_LON)**2
iy, ix = np.unravel_index(np.argmin(dist), dist.shape)
GLAT, GLON = float(lat[iy, ix]), float(lon[iy, ix])
print(f"Argonne gridcell iy={iy} ix={ix}  lat={GLAT:.4f} lon={GLON:.4f}")

## Extract point time series (CAPE, SRH, composite reflectivity)

In [ ]:
rows = []
for h in FHOURS:
    cape = open_field(h, ":CAPE:surface:")["cape"].values[iy, ix]
    srh  = open_field(h, ":HLCY:3000-0 m above ground:")["hlcy"].values[iy, ix]
    refc = open_field(h, ":REFC:entire atmosphere:")["refc"].values[iy, ix]
    valid = RUN_INIT + pd.Timedelta(hours=h)
    rows.append(dict(fxx=h, valid_utc=valid, valid_cdt=valid - CDT_OFFSET,
                     cape_jkg=float(cape), srh_0_3km_m2s2=float(srh), refc_dbz=float(refc)))
ts = pd.DataFrame(rows)
ts.to_csv("../assets/argonne_hrrr_timeseries.csv", index=False)

MAXF = int(ts.loc[ts["cape_jkg"].idxmax(), "fxx"])
valid_utc = RUN_INIT + pd.Timedelta(hours=MAXF)
valid_cdt = valid_utc - CDT_OFFSET
print(f"Max CAPE = {ts['cape_jkg'].max():.0f} J/kg at F{MAXF:02d} "
      f"(valid {valid_utc:%Y-%m-%d %H}Z = {valid_cdt:%a %b %d %H%M} CDT)")
ts.head()

## Map fields: CAPE + SRH on the domain (all hours, for the animation)

In [ ]:
# Bounding-box indices for the domain
inbox = ((lat >= DOMAIN["lat_min"]) & (lat <= DOMAIN["lat_max"]) &
         (lon >= DOMAIN["lon_min"]) & (lon <= DOMAIN["lon_max"]))
ys, xs = np.where(inbox); y0, y1, x0, x1 = ys.min(), ys.max()+1, xs.min(), xs.max()+1
DLAT, DLON = lat[y0:y1, x0:x1], lon[y0:y1, x0:x1]

cape_map, srh_map = {}, {}
for h in FHOURS:
    cape_map[h] = open_field(h, ":CAPE:surface:")["cape"].values[y0:y1, x0:x1]
    srh_map[h]  = open_field(h, ":HLCY:3000-0 m above ground:")["hlcy"].values[y0:y1, x0:x1]
print("domain grid:", DLAT.shape, "| F%02d CAPE max %d SRH max %d" % (MAXF, cape_map[MAXF].max(), srh_map[MAXF].max()))

## Skew-T profile builder (pressure levels + surface)

In [ ]:
def build_profile(h):
    '''Vertical profile at the Argonne gridcell for forecast hour h.'''
    dp = open_field(h, r":(TMP|DPT|UGRD|VGRD|HGT):\d+ mb:", product="prs")
    if isinstance(dp, list):
        import xarray as xr; dp = xr.merge(dp, compat="override")
    p = dp["isobaricInhPa"].values.astype(float); order = np.argsort(-p); p = p[order]
    T  = dp["t"].values[order, iy, ix] - 273.15
    Td = dp["dpt"].values[order, iy, ix] - 273.15
    u  = dp["u"].values[order, iy, ix]; v = dp["v"].values[order, iy, ix]
    # surface thermodynamics
    sd = open_field(h, r":(TMP:2 m above ground|DPT:2 m above ground|PRES:surface):")
    sfcP = T2 = Td2 = None
    for ds in (sd if isinstance(sd, list) else [sd]):
        if "sp"  in ds: sfcP = float(ds["sp"].values[iy, ix]) / 100.0
        if "t2m" in ds: T2   = float(ds["t2m"].values[iy, ix]) - 273.15
        if "d2m" in ds: Td2  = float(ds["d2m"].values[iy, ix]) - 273.15
    mask = p < sfcP
    P  = np.concatenate([[sfcP], p[mask]])
    TT = np.concatenate([[T2], T[mask]]); Tdd = np.concatenate([[Td2], Td[mask]])
    uu = np.concatenate([[u[mask][0] if mask.any() else u[0]], u[mask]])
    vv = np.concatenate([[v[mask][0] if mask.any() else v[0]], v[mask]])
    Pq, Tq, Tdq = P*units.hPa, TT*units.degC, Tdd*units.degC
    parcel = mpcalc.parcel_profile(Pq, Tq[0], Tdq[0]).to("degC")
    try:
        sbcape, sbcin = mpcalc.surface_based_cape_cin(Pq, Tq, Tdq)
        mucape, mucin = mpcalc.most_unstable_cape_cin(Pq, Tq, Tdq)
        sbcape, sbcin, mucape, mucin = float(sbcape.m), float(sbcin.m), float(mucape.m), float(mucin.m)
    except Exception:
        sbcape = sbcin = mucape = mucin = np.nan
    return dict(p=P, T=TT, Td=Tdd, u=uu, v=vv, parcelT=parcel.m,
                sbcape=sbcape, sbcin=sbcin, mucape=mucape, mucin=mucin)

prof = build_profile(MAXF)
print(f"F{MAXF:02d}: SBCAPE {prof['sbcape']:.0f}  SBCIN {prof['sbcin']:.0f}  "
      f"MUCAPE {prof['mucape']:.0f}  MUCIN {prof['mucin']:.0f} J/kg")

## Composite figure

In [ ]:
def draw_composite(fig, h, prof_h):
    '''Render the 4-panel composite for forecast hour h into `fig`.'''
    vutc = RUN_INIT + pd.Timedelta(hours=h); vcdt = vutc - CDT_OFFSET
    tcdt = ts["valid_cdt"]
    cape_s = gaussian_filter(cape_map[h], sigma=0.6)
    srh_s  = gaussian_filter(srh_map[h],  sigma=1.5)

    outer = gridspec.GridSpec(2, 1, height_ratios=[0.92, 1.0], hspace=0.22, figure=fig)
    axm = fig.add_subplot(outer[0], projection=ccrs.PlateCarree())
    axm.set_extent([DOMAIN["lon_min"], DOMAIN["lon_max"], DOMAIN["lat_min"], DOMAIN["lat_max"]], crs=ccrs.PlateCarree())
    axm.add_feature(USCOUNTIES.with_scale("5m"), edgecolor="0.7", linewidth=0.25, zorder=1)
    axm.add_feature(cfeature.STATES.with_scale("10m"), edgecolor="black", linewidth=1.1, zorder=3)
    axm.add_feature(cfeature.LAKES.with_scale("10m"), facecolor="#cfe8f3", edgecolor="0.4", linewidth=0.5, zorder=2)
    cf = axm.contourf(DLON, DLAT, cape_s, levels=np.arange(0, 4001, 250), cmap=CMAP_CAPE,
                      extend="max", transform=ccrs.PlateCarree(), zorder=1.5, alpha=0.9)
    cs = axm.contour(DLON, DLAT, srh_s, levels=np.arange(100, 701, 100), colors="black",
                     linewidths=1.0, transform=ccrs.PlateCarree(), zorder=4)
    axm.clabel(cs, fmt="%d", fontsize=8, inline=True)
    axm.plot(GLON, GLAT, marker="*", ms=20, mfc="white", mec="black", mew=1.4, transform=ccrs.PlateCarree(), zorder=5)
    axm.annotate("Argonne", (GLON, GLAT), xytext=(7, 6), textcoords="offset points", transform=ccrs.PlateCarree(),
                 fontsize=9, fontweight="bold", zorder=6, bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))
    for nm, (la, lo) in {"IL":(40.6,-89.6),"IA":(41.6,-91.4),"WI":(42.9,-90.0),"IN":(40.6,-86.2),"MI":(43.4,-85.2)}.items():
        axm.text(lo, la, nm, fontsize=11, fontweight="bold", color="0.2", ha="center", va="center",
                 transform=ccrs.PlateCarree(), zorder=6, alpha=0.55)
    axm.legend([Line2D([0],[0],color="black",lw=1.0)], ["0–3 km SRH (m² s⁻²)"], loc="lower left", framealpha=0.85)
    cb = fig.colorbar(cf, ax=axm, shrink=0.9, pad=0.015, ticks=np.arange(0, 4001, 500)); cb.set_label("Surface CAPE (J kg$^{-1}$)")
    axm.set_title(f"HRRR surface CAPE + 0–3 km SRH · valid {vcdt:%a %b %d} {vcdt:%H%M} CDT ({vutc:%H}Z {vutc:%b %d})\n"
                  f"init {RUN_INIT:%Y-%m-%d %H}Z run · F{h:02d}", loc="left")
    axm.text(0.012, 0.94, "a", transform=axm.transAxes, fontsize=13, fontweight="bold")

    inner = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=outer[1], wspace=0.42)
    # (b) CAPE & SRH
    axb = fig.add_subplot(inner[0]); axb2 = axb.twinx()
    l1, = axb.plot(tcdt, ts["cape_jkg"], color=C_CAPE, lw=1.8)
    l2, = axb2.plot(tcdt, ts["srh_0_3km_m2s2"], color=C_SRH, lw=1.8)
    axb.axvline(vcdt, color="0.4", ls="--", lw=1.0)
    axb.set_ylabel("CAPE (J kg$^{-1}$)", color=C_CAPE); axb.tick_params(axis="y", colors=C_CAPE)
    axb2.set_ylabel("0–3 km SRH (m² s$^{-2}$)", color=C_SRH); axb2.tick_params(axis="y", colors=C_SRH)
    axb.set_title("CAPE & SRH at Argonne", loc="left"); axb.set_xlabel("Time (CDT)")
    axb.legend([l1, l2], ["CAPE", "0–3 km SRH"], loc="upper left", framealpha=0.85)
    axb.annotate(f"max CAPE {ts['cape_jkg'].max():.0f}\n{valid_cdt:%b %d %H%M} CDT", (valid_cdt, ts["cape_jkg"].max()),
                 xytext=(14, -40), textcoords="offset points", fontsize=8, color="0.3", ha="left",
                 arrowprops=dict(arrowstyle="->", color="0.5", lw=0.8))
    axb.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H")); axb.xaxis.set_major_locator(mdates.HourLocator(interval=12))
    axb.text(-0.28, 1.02, "b", transform=axb.transAxes, fontsize=13, fontweight="bold")
    # (c) Skew-T
    skew = SkewT(fig, rotation=45, subplot=inner[1])
    P = prof_h["p"]*units.hPa; T = prof_h["T"]*units.degC; Td = prof_h["Td"]*units.degC
    skew.plot(P, T, "r", lw=1.8); skew.plot(P, Td, "g", lw=1.8)
    skew.plot(P, prof_h["parcelT"]*units.degC, "k", lw=1.2, ls="--")
    skew.shade_cape(P, T, prof_h["parcelT"]*units.degC); skew.shade_cin(P, T, prof_h["parcelT"]*units.degC)
    idx = np.arange(0, len(prof_h["p"]), 2)
    skew.plot_barbs(P[idx], (prof_h["u"]*units("m/s"))[idx], (prof_h["v"]*units("m/s"))[idx])
    skew.ax.set_ylim(1000, 150); skew.ax.set_xlim(-30, 40)
    skew.plot_dry_adiabats(alpha=0.25, lw=0.6); skew.plot_moist_adiabats(alpha=0.25, lw=0.6); skew.plot_mixing_lines(alpha=0.25, lw=0.6)
    skew.ax.set_xlabel("Temperature (°C)"); skew.ax.set_ylabel("Pressure (hPa)")
    skew.ax.set_title(f"Skew-T at Argonne · {vcdt:%b %d %H%M} CDT\n"
                      f"MUCAPE {prof_h['mucape']:.0f} · MUCIN {prof_h['mucin']:.0f} J kg$^{{-1}}$", loc="left")
    skew.ax.text(-0.18, 1.02, "c", transform=skew.ax.transAxes, fontsize=13, fontweight="bold")
    # (d) reflectivity
    axd = fig.add_subplot(inner[2])
    refc = ts["refc_dbz"].clip(lower=-10)
    axd.fill_between(tcdt, -10, refc, color=C_REFC, alpha=0.25); axd.plot(tcdt, refc, color=C_REFC, lw=1.8)
    axd.axvline(vcdt, color="0.4", ls="--", lw=1.0)
    axd.set_ylabel("Composite reflectivity (dBZ)"); axd.set_xlabel("Time (CDT)")
    axd.set_title("Composite reflectivity at Argonne", loc="left")
    axd.set_ylim(-12, max(40, refc.max()+5))
    axd.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H")); axd.xaxis.set_major_locator(mdates.HourLocator(interval=12))
    axd.text(-0.28, 1.02, "d", transform=axd.transAxes, fontsize=13, fontweight="bold")
    return fig

### Static composite at the max-CAPE hour

In [ ]:
fig = plt.figure(figsize=(14, 12.2))
draw_composite(fig, MAXF, prof)
fig.savefig("../assets/chicagoland_cape_sreh_composite.png", dpi=200, bbox_inches="tight")
plt.show()

## Animation over all forecast hours

Steps the full composite through F00–F48. Each frame refetches that hour's Skew-T
profile (cached after first pull). Writes an animated GIF.

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter

# Pre-build all profiles (network-bound; parallelize with a thread pool).
import concurrent.futures as cf
def _one(h):
    try:    return h, build_profile(h)
    except Exception as e: return h, None
profiles = {}
with cf.ThreadPoolExecutor(max_workers=6) as ex:
    for h, pr in ex.map(_one, FHOURS):
        profiles[h] = pr
profiles = {h: p for h, p in profiles.items() if p is not None}
frames = sorted(profiles)
print("profiles ready:", len(frames))

In [ ]:
anim_fig = plt.figure(figsize=(14, 12.2))
def _update(h):
    anim_fig.clf()
    draw_composite(anim_fig, h, profiles[h])
    return anim_fig.axes
anim = FuncAnimation(anim_fig, _update, frames=frames, interval=350)
anim.save("../assets/chicagoland_cape_sreh_animation.gif", writer=PillowWriter(fps=3), dpi=90)
plt.close(anim_fig)
print("wrote chicagoland_cape_sreh_animation.gif")

In [ ]:
from IPython.display import Image
Image(filename="../assets/chicagoland_cape_sreh_animation.gif")